# InternVLA-N1 Model Inferring Notebook

This notebook is used to infer the InternVLA-N1 model's by reading images and instructions from local folders. If you'd like to test the model in real world or self-built dataset, you could follow this tutorial without large **datasets download** and **simulation environment setup** (isaac-sim or habitat). Let's start!

## 0. Preparation
### 0.0 Create Conda Environment
First, we should create a conda environment through `conda create -n internvla python=3.9` and launch the jupyter kernel using the created environment. In the following tutorial, we assume the environment name is `internvla`. 

In [ ]:
%pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
import torch
print(torch.__version__)

We recommend to install flash-attn2 via pre-built wheel. If you have trouble with the installation, you might also skip this installation and remove the line of `attn_implementation="flash_attention_2"` in the model initialization.

In [ ]:
!wget https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu12torch2.6cxx11abiFALSE-cp39-cp39-linux_x86_64.whl
%pip install flash_attn-2.7.3+cu12torch2.6cxx11abiFALSE-cp39-cp39-linux_x86_64.whl 

In [ ]:
%pip install transformers==4.51.0 diffusers==0.31.0 accelerate==1.10.1 opencv-python==4.10.0.82 pillow==10.4.0 numpy==1.26.4 gym==0.23.1
%pip install imageio==2.37.0 imageio-ffmpeg==0.6.0 ftfy==6.3.1
%pip install scipy matplotlib
%pip install -e ../../. # install InternNav 

### 0.1 Prepare the dataset for inference

In [1]:
!tar -xvf ../../assets/realworld_sample_data.tar.gz -C ../../assets/

realworld_sample_data1/
realworld_sample_data1/debug_raw_0114.jpg
realworld_sample_data1/debug_raw_0111.jpg
realworld_sample_data1/debug_raw_0125_look_down.jpg
realworld_sample_data1/debug_raw_0010_look_down.jpg
realworld_sample_data1/debug_raw_0108.jpg
realworld_sample_data1/debug_raw_0095.jpg
realworld_sample_data1/debug_raw_0065.jpg
realworld_sample_data1/debug_raw_0048.jpg
realworld_sample_data1/debug_raw_0037.jpg
realworld_sample_data1/debug_raw_0106.jpg
realworld_sample_data1/debug_raw_0044.jpg
realworld_sample_data1/debug_raw_0123.jpg
realworld_sample_data1/debug_raw_0053.jpg
realworld_sample_data1/debug_raw_0017.jpg
realworld_sample_data1/debug_raw_0064.jpg
realworld_sample_data1/debug_raw_0063.jpg
realworld_sample_data1/debug_raw_0035.jpg
realworld_sample_data1/debug_raw_0067.jpg
realworld_sample_data1/debug_raw_0019.jpg
realworld_sample_data1/debug_raw_0027.jpg
realworld_sample_data1/debug_raw_0127.jpg
realworld_sample_data1/debug_raw_0082.jpg
realworld_sample_data1/debug_raw

### 0.2 Download checkpoint
The size of checkpoint is about 8GB.

In [2]:
!mkdir -p checkpoints && cd checkpoints && git clone https://huggingface.co/InternRobotics/InternVLA-N1-DualVLN
!git lfs pull

Cloning into 'InternVLA-N1-DualVLN'...
remote: Enumerating objects: 20, done.
remote: Total 20 (delta 0), reused 0 (delta 0), pack-reused 20 (from 1)
Receiving objects: 100% (20/20), 1.68 MiB | 1.10 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Error downloading object: model-00001-of-00004.safetensors (20568d1): Smudge error: Error downloading model-00001-of-00004.safetensors (20568d171de945f90a639660269e95b683fa7d3bfbe0d289061def253bbf25fa): LFS: Authorization error: https://cas-bridge.xethub.hf.co/xet-bridge-us/6938f20e3040514e86514684/491ffb6b98e374c4e1a5aec5ef652dc276af840fc66b8dc47e6f9728f2f53ae6?Expires=1780467551&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY5MzhmMjBlMzA0MDUxNGU4NjUxNDY4NC80OTFmZmI2Yjk4ZTM3NGM0ZTFhNWFlYzVlZjY1MmRjMjc2YWY4NDBmYzY2YjhkYzQ3ZTZmOTcyOGYyZjUzYWU2KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MDQ2NzU1MX19fV19&Signature=MEYCIQD3WljlGWOj2Pr%7ENZAP-n5jctqezz6LgzQCJjbxa-9y

### 0.3 Download the DepthAnything checkpoint
Download the depthanything checkpoint from [DepthAnything](https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Small) and move it into `scripts/eval/checkpoints/checkpoints` folder.

## 1. Import Required Libraries
If you meet the error about the `No module named LongCLIP (or diffusion policy)`, you should run the `git submodule update --init` in the root directory of InternNav. 

In [1]:
import sys
import os
import glob
from pathlib import Path

import numpy as np
from PIL import Image
import torch

# Add project path
project_root = Path('../../')
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src/diffusion-policy'))
#from diffusion_policy.model.diffusion.positional_embedding import SinusoidalPosEmb

print(project_root)
from internnav.agent.internvla_n1_agent_realworld import InternVLAN1AsyncAgent

../..
PROJECT_ROOT_PATH:/home/khang/Documents/VR/InternNav


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pathlib.PosixPath

## 2. Configure Parameters
Change the checkpoint path of InternVLA-N1 to the exact path in your computer. In real-world experiment, too fast inference will lead to overly close the memory intervals of the model, resulting in a large sim-to-real gap. Therefore, we use an argument `plan_step_gap` to make the model only infer every `plan_step_gap` frames when outputing trajectories. 

In [2]:
import yaml

camera_data = {
    "camera_matrix": {
        "rows": 3,
        "cols": 3,
        "data": [
            386.5, 0.0, 328.9,
            0.0, 386.5, 244.0,
            0.0, 0.0, 1.0
        ]
    },
    "image_width": 640,
    "image_height": 480
}

with open("camera_intrinsic_realworld_sample_data.yaml", "w") as f:
    yaml.dump(camera_data, f)

In [2]:
class Args:
    def __init__(self):
        self.device = "cuda:0"
        self.model_path = "/home/khang/Documents/VR/InternNav/checkpoints/InternVLA-N1-w-NavDP"
        self.resize_w = 384
        self.resize_h = 384
        self.num_history = 8
        self.camera_intrinsic = np.array([
            [386.5, 0.0, 328.9, 0.0],
            [0.0, 386.5, 244.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0]
        ])
        self.plan_step_gap = 4

args = Args()
print(f"Model path: {args.model_path}")
print(f"Device: {args.device}")
print(f"Image size: {args.resize_w}x{args.resize_h}")
print(f"History frames: {args.num_history}")

Model path: /home/khang/Documents/VR/InternNav/checkpoints/InternVLA-N1-w-NavDP
Device: cuda:0
Image size: 384x384
History frames: 8


## 3. Initialize Agent
Load the InternVLA-N1 model and initialize the agent. If you meet error about transformers, please check that the `flash_attn` and `accelerate` is correctly installed.

In [3]:
print("Loading model...")
agent = InternVLAN1AsyncAgent(args)

# Warm up model
print("Warming up model...")
dummy_rgb = np.zeros((480, 640, 3), dtype=np.uint8)
dummy_depth = np.zeros((480, 640), dtype=np.float32)
dummy_pose = np.eye(4)
agent.reset()
agent.step(dummy_rgb, dummy_depth, dummy_pose, "hello", intrinsic=args.camera_intrinsic)
print("Model loaded successfully!")

Loading model...
args.model_path/home/khang/Documents/VR/InternNav/checkpoints/InternVLA-N1-w-NavDP


xFormers not available
xFormers not available
/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/torch/nn/modules/module.py:2588: UserWarning: for pretrained.cls_token: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/torch/nn/modules/module.py:2588: UserWarning: for pretrained.pos_embed: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/torch/nn/modules/

Warming up model...


/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


output 1  ←←←← cost: 1.4177980422973633s
Model loaded successfully!


## 4. Configure Test Data Path
Now we used a pre-collected real-world dataset to test our model. The images were captured through a Unitree Go2 robot mounted with a realsense D455. You could freely change the dataset to your own dataset and an `instruction.txt` file. Note that the `InternVLA-N1` model use depth image inputs for S1 model, but we forgot recording the depth image in real-world dataset. If you want to build your own dataset, please record **both the aligned depth and rgb images**. 

In [6]:
# Configure data directory (single scene per folder)
scene_dir = '../../assets/realworld_sample_data1'

# Check if instruction file exists
instruction_path = os.path.join(scene_dir, 'instruction.txt')
if not os.path.exists(instruction_path):
    print(f"Error: instruction.txt not found in {scene_dir}")
else:
    print(f"Scene directory: {scene_dir}")
    
    # Read instruction
    with open(instruction_path, 'r') as f:
        instruction = f.read().strip()
    print(f"Instruction: {instruction}")
    
    # Get all debug_raw images
    rgb_paths = sorted(glob.glob(os.path.join(scene_dir, 'debug_raw_*.jpg')))
    print(f"\nFound {len(rgb_paths)} images")
    # Show first few image names
    print("\nFirst 5 images:")
    for i, path in enumerate(rgb_paths[:5]):
        print(f"  {i+1}. {os.path.basename(path)}")

Scene directory: ../../assets/realworld_sample_data1
Instruction: Turn around and walk out of this office. Turn towards your slight right at the chair. Move forward to the walkway and go near the red bin. You can see an open door on your right side, go inside the open door. Stop at the computer monitor.

Found 152 images

First 5 images:
  1. debug_raw_0000.jpg
  2. debug_raw_0001.jpg
  3. debug_raw_0002.jpg
  4. debug_raw_0003.jpg
  5. debug_raw_0004.jpg


Now add some visualization function for the model.

In [7]:
from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg


def annotate_image(idx, image, llm_output=None, trajectory=None, pixel_goal=None, output_dir="./"):
    os.makedirs(output_dir, exist_ok=True)

    image = Image.fromarray(image.astype(np.uint8)).convert("RGB")
    draw = ImageDraw.Draw(image)

    font_size = 20
    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", font_size)
    except:
        font = ImageFont.load_default()

    text_content = [
        f"Frame Id : {idx}",
        f"Output   : {llm_output}",
    ]

    max_width = 0
    total_height = 0
    line_height = 26

    for line in text_content:
        bbox = draw.textbbox((0, 0), line, font=font)
        text_width = bbox[2] - bbox[0]
        max_width = max(max_width, text_width)
        total_height += line_height

    padding = 10
    box_x, box_y = 10, 10
    box_width = max_width + 2 * padding
    box_height = total_height + 2 * padding

    draw.rectangle(
        [box_x, box_y, box_x + box_width, box_y + box_height],
        fill="black"
    )

    y_position = box_y + padding
    for line in text_content:
        draw.text((box_x + padding, y_position), line, fill="white", font=font)
        y_position += line_height

    image = np.array(image)

    # Draw trajectory if available
    if trajectory is not None and len(trajectory) > 0:
        img_height, img_width = image.shape[:2]

        window_size = 200
        window_x = max(0, img_width - window_size)
        window_y = 0

        traj_points = []
        for point in trajectory:
            if isinstance(point, (list, tuple, np.ndarray)) and len(point) >= 2:
                traj_points.append([float(point[0]), float(point[1])])

        if len(traj_points) > 0:
            traj_array = np.array(traj_points)
            x_coords = traj_array[:, 0]
            y_coords = traj_array[:, 1]

            fig, ax = plt.subplots(figsize=(2, 2), dpi=100)
            fig.patch.set_alpha(0.6)
            fig.patch.set_facecolor("gray")
            ax.set_facecolor("lightgray")

            ax.plot(y_coords, x_coords, "b-", linewidth=2)
            ax.plot(y_coords[0], x_coords[0], "go", markersize=6)
            ax.plot(y_coords[-1], x_coords[-1], "ro", markersize=6)
            ax.plot(0, 0, "w+", markersize=10, markeredgewidth=2)

            ax.set_xlabel("Y", fontsize=8)
            ax.set_ylabel("X", fontsize=8)
            ax.invert_xaxis()
            ax.tick_params(labelsize=6)
            ax.grid(True, alpha=0.3, linewidth=0.5)
            ax.set_aspect("equal", adjustable="box")

            plt.tight_layout(pad=0.3)

            canvas = FigureCanvasAgg(fig)
            canvas.draw()
            plot_img = np.asarray(canvas.buffer_rgba())[:, :, :3].copy()
            plt.close(fig)

            plot_img = cv2.resize(plot_img, (window_size, window_size))

            image[
                window_y:window_y + window_size,
                window_x:window_x + window_size
            ] = plot_img

    # Draw pixel goal if available
    if pixel_goal is not None:
        pixel_goal = np.asarray(pixel_goal).astype(int)

        # Usually pixel_goal is [row, col], so cv2 needs (col, row)
        row, col = int(pixel_goal[0]), int(pixel_goal[1])

        h, w = image.shape[:2]
        if 0 <= row < h and 0 <= col < w:
            cv2.circle(image, (col, row), 6, (255, 0, 0), -1)

    save_path = os.path.join(output_dir, f"rgb_{idx}_annotated.png")
    Image.fromarray(image).convert("RGB").save(save_path)

    return image

## 5. Run Model Testing
We begin to read the local images, instruction to run the model. Please make sure that the depth image is fed to the model and **the unit is in meter**. You could print the maximum value of the depth image in your real-world experiment and check the value. 

If everything goes well, the model will rotate in place at the begining. Then it generates the correct pixel goal and trajectories. The visualization results are also saved in the `save_dir` folder.

In [8]:
# Reset agent
agent.reset()
print(f"{'='*80}")
print(f"Processing scene: {os.path.basename(scene_dir)}")
print(f"Instruction: '{instruction}'")
print(f"Total images: {len(rgb_paths)}")
print(f"{'='*80}\n")

action_seq = []
look_down = False

save_dir = '../../test_data/'
os.makedirs(save_dir, exist_ok=True)
# Process each image
for i, rgb_path in enumerate(rgb_paths):
    # Check if this is a look_down image
    look_down = ('look_down' in rgb_path)
    
    # Extract image ID from filename (e.g., debug_raw_0003.jpg -> 0003)
    basename = os.path.basename(rgb_path)
    if look_down:
        # e.g., debug_raw_0010_look_down.jpg -> 0010
        image_id = basename.replace('debug_raw_', '').replace('_look_down.jpg', '')
    else:
        # e.g., debug_raw_0003.jpg -> 0003
        image_id = basename.replace('debug_raw_', '').replace('.jpg', '')
        
    # Read RGB image
    rgb = np.asarray(Image.open(rgb_path).convert('RGB'))
    
    # Create dummy depth image (not available in test data)
    # !Note You must full in depth to model
    depth = 10 * np.ones((rgb.shape[0], rgb.shape[1]), dtype=np.float32)
    
    # Create dummy camera pose
    camera_pose = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])
    
    # Run model or just save image
    # print(f"[{i+1}/{len(rgb_paths)}] Running model inference: {os.path.basename(rgb_path)}")
    with torch.no_grad():   
        dual_sys_output = agent.step(
            rgb, 
            depth, 
            camera_pose, 
            instruction, 
            intrinsic=args.camera_intrinsic,
            look_down=look_down
        )
    
    # Print output results and save every frame
    output_action = dual_sys_output.output_action
    output_traj = dual_sys_output.output_trajectory
    output_pixel = dual_sys_output.output_pixel

    if output_action is not None and output_action != []:
        print(f"Output action: {output_action}")
        llm_output = output_action

        # Still save this image even if there is no trajectory/pixel
        annotate_image(
            image_id,
            rgb,
            llm_output=llm_output,
            trajectory=None,
            pixel_goal=None,
            output_dir=save_dir
        )

    else:
        if output_traj is not None:
            try:
                trajectory = output_traj.tolist()
            except:
                trajectory = output_traj
            print(f"output_trajectory: {trajectory}")
        else:
            trajectory = None
            print("output_trajectory: None")

        if output_pixel is not None:
            print(f"output_pixel: {output_pixel}")

        annotate_image(
            image_id,
            rgb,
            llm_output="traj",
            trajectory=trajectory,
            pixel_goal=output_pixel,
            output_dir=save_dir
        )

print(f"\nScene {os.path.basename(scene_dir)} completed!")


Processing scene: realworld_sample_data1
Instruction: 'Turn around and walk out of this office. Turn towards your slight right at the chair. Move forward to the walkway and go near the red bin. You can see an open door on your right side, go inside the open door. Stop at the computer monitor.'
Total images: 152



/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/khang/miniconda3/envs/internnav/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


output 1  →→→→ cost: 0.7870099544525146s
Output action: [3, 3, 3, 3]
output 2  →→→→ cost: 0.9828789234161377s
Output action: [3, 3, 3, 3]
output 3  →→→→ cost: 1.3207244873046875s
Output action: [3, 3, 3, 3]
output 4  →→→→ cost: 1.5025112628936768s
Output action: [3, 3, 3, 3]
output 5  →→→→ cost: 1.8307933807373047s
Output action: [3, 3, 3, 3]
output 6  →→→→ cost: 2.247652053833008s
Output action: [3, 3, 3, 3]
output 7  →→→→ cost: 2.624640464782715s
Output action: [3, 3, 3, 3]
output 8  →→→→ cost: 2.8878045082092285s
Output action: [3, 3, 3, 3]
output 9  →→→→ cost: 3.3238768577575684s
Output action: [3, 3, 3, 3]
output 10  ↓ cost: 2.9194791316986084s
Output action: [5]
output 11  ↓ cost: 2.9445667266845703s
Output action: [5]
output 11  461 208 cost: 4.527441740036011s
output_trajectory: [[0.0, 0.0], [0.1038665771484375, -0.008717909455299377], [0.2070770263671875, -0.023324057459831238], [0.29717254638671875, -0.04226817190647125], [0.37898164987564087, -0.06389321386814117], [0.454494

# 6.Visualize Results
It's worth noting that we input an zero depth image to the model, so the output trajectories are short. In your own experiments, please check the output lengths of the model are about **2m**. If not, you should check the model inputs or create au issue on Github.

In [ ]:

import glob
from PIL import Image
import matplotlib.pyplot as plt

for img_path in sorted(glob.glob(f'{save_dir}/*_annotated.png')):
    plt.imshow(Image.open(img_path))
    plt.axis('off')
    plt.show()

In [10]:
import glob
import cv2
from PIL import Image
import numpy as np
import os

image_files = sorted(glob.glob(f"{save_dir}/*_annotated.png"))

assert len(image_files) > 0, "No annotated images found"

# Read first frame to get size
first_frame = cv2.imread(image_files[0])
height, width = first_frame.shape[:2]

fps = 2  # adjust as desired

video_path = os.path.join(save_dir, "trajectory_video.mp4")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(
    video_path,
    fourcc,
    fps,
    (width, height)
)

for img_path in image_files:
    frame = cv2.imread(img_path)
    video_writer.write(frame)

video_writer.release()

print(f"Saved video to: {video_path}")

Saved video to: ../../test_data/trajectory_video.mp4


In [1]:
from __future__ import annotations

import argparse
import glob
import os
import sys
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.backends.backend_agg import FigureCanvasAgg
from PIL import Image, ImageDraw, ImageFont

# TODO:
# later fix to deoth anything checkpoint path input
# later fix to cam intrinsic file
# data look like?


# =============================================================================
# USER CONFIGURATION — edit these defaults before running
# =============================================================================

# InternNav repo root (directory that contains the `internnav` package).
# The notebook uses `../../` relative to scripts/notebooks/.
PROJECT_ROOT = Path(__file__).resolve().parent.parent / "InternNav"


NameError: name '__file__' is not defined